# **02 — Brasileirão Série A 2018–2025: paradoxo do aniversário**

Para cada partida completa (22 titulares + 1 reserva sorteado, todos com data de nascimento conhecida), verifica se pelo menos dois dos 23 jogadores compartilham o mesmo dia/mês de nascimento — o paradoxo do aniversário aplicado a escalações reais — e compara a proporção observada com a probabilidade teórica clássica (365 dias uniformes).

Depende dos arquivos gerados pelo notebook **01**.

## **1. Leitura dos dados coletados**

In [1]:
import pandas as pd
from math import prod, sqrt
from pathlib import Path

from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "brasileirao_serie_a_2018_2025"
ANALYSIS_DIR = PROJECT_DIR / "data" / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

COVERAGE_PATH = DATA_DIR / "brasileirao_serie_a_2018_2025_extraction_coverage.csv"
SELECTED_PLAYERS_PATH = DATA_DIR / "brasileirao_serie_a_2018_2025_selected_23_players.csv"
for path in (COVERAGE_PATH, SELECTED_PLAYERS_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Arquivo ausente: {path}. Execute o notebook 01 (coleta de dados).")

coverage_df = pd.read_csv(COVERAGE_PATH, dtype={"game_id": "string"})
coverage_df["date"] = pd.to_datetime(coverage_df["date"], errors="coerce").dt.date

selected_players_df = pd.read_csv(
    SELECTED_PLAYERS_PATH, dtype={"game_id": "string", "player_id": "string"}
)
selected_players_df["birth_date"] = pd.to_datetime(
    selected_players_df["birth_date"], errors="coerce"
)
selected_players_df["date"] = pd.to_datetime(
    selected_players_df["date"], errors="coerce"
).dt.date

complete_ids = set(coverage_df.loc[coverage_df["status"] == "complete", "game_id"])
analysis_players_df = selected_players_df.loc[
    selected_players_df["game_id"].isin(complete_ids)
].copy()

quality = analysis_players_df.groupby("game_id").agg(
    selected=("player_id", "size"), known=("birth_date", "count"),
    starters=("role", lambda values: (values == "titular").sum()),
    reserve=("role", lambda values: (values == "reserva_sorteado").sum()),
)
assert quality.selected.eq(23).all()
assert quality.known.eq(23).all()
assert quality.starters.eq(22).all()
assert quality.reserve.eq(1).all()

print(f"Partidas completas para análise: {len(complete_ids)} de {len(coverage_df)}")

Partidas completas para análise: 3026 de 3040


## **2. Paradoxo do aniversário por temporada**

In [2]:
analysis_players_df["birth_day_month"] = pd.to_datetime(
    analysis_players_df["birth_date"]
).dt.strftime("%d/%m")
birthday_groups_df = (
    analysis_players_df.groupby(["game_id", "birth_day_month"])
    .size().rename("players_on_birthday").reset_index()
)
shared_groups_df = birthday_groups_df.loc[birthday_groups_df["players_on_birthday"] >= 2].copy()
shared_groups_df["birthday_pairs"] = (
    shared_groups_df["players_on_birthday"]
    * (shared_groups_df["players_on_birthday"] - 1) // 2
)

match_results_df = coverage_df.loc[
    coverage_df["status"] == "complete",
    ["game_id", "season", "round", "date", "home_team", "away_team"],
].copy()
per_match = shared_groups_df.groupby("game_id").agg(
    shared_birthday_dates=("birth_day_month", "size"),
    birthday_pairs=("birthday_pairs", "sum"),
    largest_birthday_group=("players_on_birthday", "max"),
)
match_results_df = match_results_df.join(per_match, on="game_id")
for column in ["shared_birthday_dates", "birthday_pairs", "largest_birthday_group"]:
    match_results_df[column] = match_results_df[column].fillna(0).astype(int)
match_results_df["has_shared_birthday"] = match_results_df["shared_birthday_dates"] >= 1

shared_players_df = analysis_players_df.merge(
    shared_groups_df[["game_id", "birth_day_month", "players_on_birthday"]],
    on=["game_id", "birth_day_month"], how="inner",
)

def wilson_interval(successes, observations, z=1.96):
    proportion = successes / observations
    denominator = 1 + z**2 / observations
    center = (proportion + z**2 / (2 * observations)) / denominator
    margin = z * sqrt(
        proportion * (1 - proportion) / observations + z**2 / (4 * observations**2)
    ) / denominator
    return center - margin, center + margin


theoretical_probability = 1 - prod((365 - i) / 365 for i in range(23))

def summarize(frame):
    matches = len(frame)
    positive = int(frame["has_shared_birthday"].sum())
    probability = positive / matches
    low, high = wilson_interval(positive, matches)
    return pd.Series({
        "matches": matches, "matches_with_shared_birthday": positive,
        "observed_probability": probability, "confidence_95_low": low,
        "confidence_95_high": high, "theoretical_probability": theoretical_probability,
        "difference_percentage_points": (probability - theoretical_probability) * 100,
    })


season_summary_df = (
    match_results_df.groupby("season", sort=True)
    .apply(summarize, include_groups=False).reset_index()
)
overall_summary_df = summarize(match_results_df).to_frame().T
overall_summary_df.insert(0, "scope", "Brasileirão Série A, 2018–2025")

display(overall_summary_df)
display(season_summary_df)

,scope,matches,matches_with_shared_birthday,observed_probability,confidence_95_low,confidence_95_high,theoretical_probability,difference_percentage_points
0,"Brasileirão Série A, 2018–2025",3026.0,1717.0,0.567416,0.549689,0.584972,0.507297,6.01185


,season,matches,matches_with_shared_birthday,observed_probability,confidence_95_low,confidence_95_high,theoretical_probability,difference_percentage_points
0,2018,379.0,198.0,0.522427,0.472166,0.572239,0.507297,1.513021
1,2019,378.0,205.0,0.542328,0.491929,0.591876,0.507297,3.503081
2,2020,380.0,197.0,0.518421,0.468250,0.568224,0.507297,1.112382
3,2021,376.0,213.0,0.566489,0.515976,0.615658,0.507297,5.919213
4,2022,377.0,215.0,0.570292,0.519859,0.619306,0.507297,6.299454
5,2023,377.0,230.0,0.610080,0.559971,0.657967,0.507297,10.278234
6,2024,379.0,231.0,0.609499,0.559517,0.657282,0.507297,10.220145
7,2025,380.0,228.0,0.600000,0.549979,0.648019,0.507297,9.270277


## **3. Exportação**

In [3]:
overall_summary_df.to_csv(
    ANALYSIS_DIR / "brasileirao_serie_a_2018_2025_shared_birthday_overall.csv", index=False
)
season_summary_df.to_csv(
    ANALYSIS_DIR / "brasileirao_serie_a_2018_2025_shared_birthday_by_season.csv", index=False
)
match_results_df.to_csv(
    ANALYSIS_DIR / "brasileirao_serie_a_2018_2025_shared_birthday_match_results.csv", index=False
)
shared_players_df.to_csv(
    ANALYSIS_DIR / "brasileirao_serie_a_2018_2025_shared_birthday_players.csv", index=False
)

print(f"Probabilidade teórica: {theoretical_probability:.2%}")
print(f"Probabilidade observada: {overall_summary_df.observed_probability.iloc[0]:.2%}")
print("\nContinua no notebook 03 — por que o observado difere do teórico.")

Probabilidade teórica: 50.73%
Probabilidade observada: 56.74%

Continua no notebook 03 — por que o observado difere do teórico.
